In [13]:
import os
import sys

project_root = os.path.abspath("..")

sys.path.append(project_root)

In [14]:
from src.data_preprocessing import load_and_clean_data, normalize_zscore, filter_top_genes_by_variance
from src.metrics import calculate_euclidean_distance, calculate_pearson_distance
from src.clustering import AgglomerativeClustering
from src.visualization import plot_dendrogram, plot_heatmap

import pandas as pd
import numpy as np

In [3]:
DATA_PATH = "../data/raw/data_set_ALL_AML_independent.csv"

df_raw = load_and_clean_data(DATA_PATH)
df_norm = normalize_zscore(df_raw, axis=1)
df_subset = filter_top_genes_by_variance(df_norm, top_n=50)

df_subset.head()


2026-06-21 08:34:46,569 [INFO]: Đang đọc dữ liệu từ: ../data/raw/data_set_ALL_AML_independent.csv
2026-06-21 08:34:46,689 [INFO]: Kích thước ma trận biểu hiện: 7129 hàng (gen) × 34 cột (bệnh nhân)
2026-06-21 08:34:46,699 [INFO]: Chuẩn hóa Z-score theo gen (hàng)...
2026-06-21 08:34:46,720 [INFO]: Đã lọc Top 50 gen có phương sai cao nhất.


,39,40,42,47,48,49,41,43,44,45,...,54,57,58,60,61,65,66,63,64,62
Gene Accession Number,,,,,,,,,,,,,,,,,,,,,
M12886_at,-0.417234,0.235713,-0.346346,0.112766,-0.415019,-0.453232,-0.538519,-0.418895,-0.454893,-0.467077,...,-0.416680,-0.359637,-0.199031,-0.482030,-0.401173,-0.512490,0.422349,-0.330285,-0.127035,0.349799
Z83735_at,3.199461,-0.399480,-0.054376,-0.633657,-0.011238,-1.459441,-0.109839,-0.122164,0.124339,2.811219,...,-0.639820,0.025738,-0.639820,0.112014,-0.325529,-0.621332,-0.541219,0.512581,-1.163638,-0.584357
L28010_at,0.118634,-0.315857,-0.622556,0.371893,1.965800,0.406746,1.350078,-0.090479,0.232485,0.722739,...,3.417975,-0.738730,0.195309,0.134898,-0.246152,-0.778229,-1.019871,0.697181,-0.643467,-0.290298
M12125_at,-0.077171,-0.382152,0.092263,0.068058,2.745118,-1.035684,1.917311,1.941516,-0.406357,0.581202,...,-0.837204,-0.808158,-1.989356,-1.205118,0.406927,-0.682293,-0.779112,0.556997,0.861978,0.300425
X63187_at,2.263846,0.884786,-1.328123,-0.462202,0.339577,-1.215874,-0.638593,1.141356,0.195257,-1.873332,...,-0.542380,-0.478237,0.708395,-0.478237,-0.734807,0.580110,1.766743,0.788573,0.548039,-1.103625


In [4]:
# Patients clustering
X_patients = df_subset.values.T
model_patient = AgglomerativeClustering(linkage='average')
Z_patients = model_patient.fit(X_patients, calculate_euclidean_distance)

# Gene clustering
X_genes = df_subset.values
model_gene = AgglomerativeClustering(linkage='average')
Z_genes = model_gene.fit(X_genes, calculate_pearson_distance)

print("Hoàn thành phân cụm!")

2026-06-21 08:34:46,772 [INFO]: Bắt đầu gom cụm 34 mẫu bằng phương pháp average...
2026-06-21 08:34:46,833 [INFO]: Bắt đầu gom cụm 50 mẫu bằng phương pháp average...


Hoàn thành phân cụm!


In [5]:
gene_variance = df_norm.var(axis=1).sort_values(ascending=False)

top10_genes = gene_variance.head(10)
top10_genes

Gene Accession Number
M12886_at    1.0
Z83735_at    1.0
L28010_at    1.0
M12125_at    1.0
X63187_at    1.0
D78134_at    1.0
D90359_at    1.0
Z48510_at    1.0
U58089_at    1.0
Z21707_at    1.0
dtype: float64

In [6]:
threshold = gene_variance.mean()
high_var_genes = gene_variance[gene_variance > threshold]

len(high_var_genes), high_var_genes.head()

(1965,
 Gene Accession Number
 M12886_at    1.0
 Z83735_at    1.0
 L28010_at    1.0
 M12125_at    1.0
 X63187_at    1.0
 dtype: float64)

## Tra cứu ý nghĩa sinh học

Tra cứu các gen tại:

- NCBI Gene: https://www.ncbi.nlm.nih.gov/gene
- GeneCards: https://www.genecards.org/
- UniProt: https://www.uniprot.org/

Ví dụ:
1. Copy tên gene
2. Dán vào GeneCards
3. Xem:
   - Function
   - Disease association
   - Expression
